In [1]:
import pandas as pd
import numpy as np

In [5]:
renew = pd.read_csv("renewable-share-energy.csv")
gdp_wide = pd.read_csv("gpd_per_capita.csv", skiprows=4)

In [6]:
gdp_long = gdp_wide.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="GDP_per_capita",
)

gdp_long["Year"] = pd.to_numeric(gdp_long["Year"], errors="coerce")
gdp_long = gdp_long.dropna(subset=["Year"])
gdp_long["Year"] = gdp_long["Year"].astype(int)

In [7]:
renew_countries = renew[~renew["Code"].str.startswith("OWID", na=False)].copy()

In [8]:
NAME_MAP = {
    "Egypt":       "Egypt, Arab Rep.",
    "Iran":        "Iran, Islamic Rep.",
    "Russia":      "Russian Federation",
    "Slovakia":    "Slovak Republic",
    "South Korea": "Korea, Rep.",
    "Turkey":      "Turkiye",
    "Venezuela":   "Venezuela, RB",
    "Vietnam":     "Viet Nam",
    "Hong Kong":   "Hong Kong SAR, China",
    "Taiwan":      "Taiwan",
}

renew_countries["Entity_wb"] = renew_countries["Entity"].replace(NAME_MAP)

In [9]:
ei_aggregates = renew_countries["Entity_wb"].str.contains(r"\(EI\)", na=False)
renew_clean = renew_countries[~ei_aggregates].copy()

In [10]:
gdp_trimmed = gdp_long[gdp_long["Year"].between(1965, 2024)].copy()

In [11]:
merged = renew_clean.merge(
    gdp_trimmed[["Country Name", "Year", "GDP_per_capita"]],
    left_on=["Entity_wb", "Year"],
    right_on=["Country Name", "Year"],
    how="left",
)

In [12]:
merged["gdp_missing"] = merged["GDP_per_capita"].isna()

def missing_reason(row):
    if not row["gdp_missing"]:
        return None
    if row["Entity_wb"] == "Taiwan":
        return "Taiwan not in World Bank (non-UN member)"
    return "No GDP data for this country-year in World Bank"

merged["missing_reason"] = merged.apply(missing_reason, axis=1)

In [13]:
final = merged[[
    "Entity", "Entity_wb", "Code", "Year",
    "Renewables", "GDP_per_capita", "gdp_missing", "missing_reason",
]].rename(columns={
    "Entity":     "country_owid",
    "Entity_wb":  "country_wb",
    "Code":       "iso3",
    "Renewables": "renewables_pct",
})

final = final.sort_values(["country_owid", "Year"]).reset_index(drop=True)

final.to_csv("merged_renewable_gdp.csv", index=False)

1. Skipped first 4 rows from gdp per capita file as these were column names
2. Dropped all rows from the renewable dataset whose Code begins with OWID. OWID uses the prefix OWID for non-sovereign groupings (continents, income bands, EI regional blocs).
3. Taiwan note: Taiwan appears in the OWID dataset under the ISO code TWN, but it is not included in the World Bank dataset because the World Bank follows the UN member state list. To preserve the renewable energy data, Taiwan rows were kept with Entity_wb = "Taiwan". As a result, GDP values are missing (NaN) for all Taiwan entries.
4. Missing GDP is concentrated in:
Taiwan (60 rows), as it is not a part of UN so is absent from World Bank
Former Soviet / Eastern Bloc states (Azerbaijan, Belarus, Bulgaria, Czechia, Estonia, Hungary, Kazakhstan, Latvia, Lithuania, Poland, Romania, Russia, Slovakia, Ukraine, Uzbekistan) — World Bank data unavailable for pre-independence or early transition years
Vietnam, Indonesia, UAE, Qatar — sparse early coverage
5. Trimmed the GDP data to 1965–2024 before merging as renewable energy data is only available 1965 onwards
6. 